In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from diffusers import FluxPipeline, BitsAndBytesConfig
from transformers import T5EncoderModel
from diffusers.quantizers import PipelineQuantizationConfig
from IPython.display import display

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipeline_quant_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": torch.bfloat16},
    components_to_quantize=["transformer", "text_encoder_2"],
)

bfl_repo = "black-forest-labs/FLUX.1-schnell"
text_encoder_2 = T5EncoderModel.from_pretrained(
    bfl_repo,
    subfolder="text_encoder_2",
    torch_dtype=torch.bfloat16,
)
pipe = FluxPipeline.from_pretrained(
    bfl_repo,
    torch_dtype=torch.bfloat16,
    text_encoder_2=text_encoder_2,
    force_download=False,
    quantization_config=pipeline_quant_config,
).to(device)

In [ ]:
prompt = "taco bell"
num_inference_steps=4

image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=num_inference_steps,
    max_sequence_length=256,
    generator=torch.Generator(device).manual_seed(0)
).images[0]


In [ ]:
w, h = image.size
scale = 0.25  
display(image.resize((int(w * scale), int(h * scale))))


In [ ]:
from typing import Callable, List, Tuple
import torch.nn as nn
import functools
from contextlib import contextmanager


# module_dict = {
#     f"transformer.transformer_blocks.{i}": block
#     for i, block in enumerate(pipe.transformer.transformer_blocks)
# }
# print(f"pipe.transformer: {pipe.transformer}")
# print(f"pipe.transformer.transformer_blocks: {pipe.transformer.transformer_blocks}")
modules = [b for b in pipe.transformer.transformer_blocks]
# print("Stored modules:")
# for name in module_dict:
#     print(name, "->", module_dict[name])



T = len(pipe.transformer.transformer_blocks) # to allocate at runtime -- dependent on input length
n = pipe.transformer.transformer_blocks[0].attn.to_q.in_features
# n = pipe.transformer.config["joint_attention_dim"]
print(n)
print(T)

B = 1
X_text = torch.zeros(T, B, 1, n, device=device) # TODO: reshape properly
X_img = torch.zeros(T, B, 1, n, device=device) # TODO:

T_sing = len(pipe.transformer.single_transformer_blocks) # to allocate at runtime -- dependent on input length
X_text_sing = torch.zeros(T_sing, B, 1, n, device=device) # TODO: reshape properly
X_img_sing = torch.zeros(T_sing, B, 1, n, device=device) # TODO:

def hook_collector_multi(layer_idx, module, input, output):
    # print(f"outie: {len(output)}")
    # print(output[0].shape)
    # print(output[1].shape)
    X_text[layer_idx][..., 0,:] = output[0][-1][-1]
    X_img[layer_idx][..., 0,:] = output[1][-1][-1]
    return output

def hook_collector_single(layer_idx, module, input, output):
    # print(f"outie: {len(output)}")
    # print(output[0].shape)
    # print(output[1].shape)
    X_text_sing[layer_idx][..., 0,:] = output[0][-1][-1]
    X_img_sing[layer_idx][..., 0,:] = output[1][-1][-1]
    return output

@contextmanager
def add_hooks(
        # module_forward_hooks: List[Tuple[nn.Module, Callable]] = None,
        # **kwargs,
    ):
        """Context manager for temporarily adding forward hooks.

        Args:
            module_forward_pre_hooks: List of (module, hook_fn) tuples for pre-hooks
            module_forward_hooks: List of (module, hook_fn) tuples for forward hooks
            **kwargs: Additional keyword arguments passed to hook functions

        Yields:
            None. Hooks are active within the context, removed on exit.
        """
        # module_forward_pre_hooks = module_forward_pre_hooks or []
        # module_forward_hooks = module_forward_hooks or []
        handles = []
        try:

            for layer_idx, layer in enumerate(pipe.transformer.transformer_blocks):
                def hook_wrapper(layer_idx):
                    def hook(module, input, output):
                        return hook_collector_multi(layer_idx, module, input, output)

                    return hook

                handles.append(
                    layer.register_forward_hook(
                        hook_wrapper(layer_idx)
                    )
                )
            for layer_idx, layer in enumerate(pipe.transformer.single_transformer_blocks):
                def hook_wrapper(layer_idx):
                    def hook(module, input, output):
                        return hook_collector_single(layer_idx, module, input, output)

                    return hook

                handles.append(
                    layer.register_forward_hook(
                        hook_wrapper(layer_idx)
                    )
                )
            # for module, hook in module_forward_hooks:
                # partial_hook = functools.partial(hook, **kwargs)
                # handles.append(module.register_forward_hook(partial_hook))
            yield
        finally:
            for h in handles:
                h.remove()



In [ ]:
from transformers import T5TokenizerFast
prompt = "cool awesome futuristic robot"

tokenizer = T5TokenizerFast.from_pretrained("black-forest-labs/FLUX.1-schnell", subfolder="tokenizer_2")
tokens = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)


with add_hooks():
# add_hooks()
    image = pipe(
        prompt,
        guidance_scale=0.0,
        num_inference_steps=num_inference_steps,
        max_sequence_length=256,
        generator=torch.Generator(device).manual_seed(0)
    ).images[0]

# print(X_text)
# print(X_img)
w, h = image.size
scale = 0.25  # 25% size
display(image.resize((int(w * scale), int(h * scale))))


In [ ]:
from typing import Any, Callable, Dict, List, Optional, Union
import inspect
def retrieve_timesteps(
    scheduler,
    num_inference_steps: Optional[int] = None,
    device: Optional[Union[str, torch.device]] = None,
    timesteps: Optional[List[int]] = None,
    sigmas: Optional[List[float]] = None,
    **kwargs,
):
    r"""
    Calls the scheduler's `set_timesteps` method and retrieves timesteps from the scheduler after the call. Handles
    custom timesteps. Any kwargs will be supplied to `scheduler.set_timesteps`.

    Args:
        scheduler (`SchedulerMixin`):
            The scheduler to get timesteps from.
        num_inference_steps (`int`):
            The number of diffusion steps used when generating samples with a pre-trained model. If used, `timesteps`
            must be `None`.
        device (`str` or `torch.device`, *optional*):
            The device to which the timesteps should be moved to. If `None`, the timesteps are not moved.
        timesteps (`List[int]`, *optional*):
            Custom timesteps used to override the timestep spacing strategy of the scheduler. If `timesteps` is passed,
            `num_inference_steps` and `sigmas` must be `None`.
        sigmas (`List[float]`, *optional*):
            Custom sigmas used to override the timestep spacing strategy of the scheduler. If `sigmas` is passed,
            `num_inference_steps` and `timesteps` must be `None`.

    Returns:
        `Tuple[torch.Tensor, int]`: A tuple where the first element is the timestep schedule from the scheduler and the
        second element is the number of inference steps.
    """
    if timesteps is not None and sigmas is not None:
        raise ValueError("Only one of `timesteps` or `sigmas` can be passed. Please choose one to set custom values")
    if timesteps is not None:
        accepts_timesteps = "timesteps" in set(inspect.signature(scheduler.set_timesteps).parameters.keys())
        if not accepts_timesteps:
            raise ValueError(
                f"The current scheduler class {scheduler.__class__}'s `set_timesteps` does not support custom"
                f" timestep schedules. Please check whether you are using the correct scheduler."
            )
        scheduler.set_timesteps(timesteps=timesteps, device=device, **kwargs)
        timesteps = scheduler.timesteps
        num_inference_steps = len(timesteps)
    elif sigmas is not None:
        accept_sigmas = "sigmas" in set(inspect.signature(scheduler.set_timesteps).parameters.keys())
        if not accept_sigmas:
            raise ValueError(
                f"The current scheduler class {scheduler.__class__}'s `set_timesteps` does not support custom"
                f" sigmas schedules. Please check whether you are using the correct scheduler."
            )
        scheduler.set_timesteps(sigmas=sigmas, device=device, **kwargs)
        timesteps = scheduler.timesteps
        num_inference_steps = len(timesteps)
    else:
        scheduler.set_timesteps(num_inference_steps, device=device, **kwargs)
        timesteps = scheduler.timesteps
    return timesteps, num_inference_steps


In [ ]:
import numpy as np
sigmas = np.linspace(1.0, 1 / num_inference_steps, num_inference_steps)
if hasattr(pipe.scheduler.config, "use_flow_sigmas") and pipe.scheduler.config.use_flow_sigmas:
    sigmas = None
# image_seq_len = X.shape[1] # TODO
image_seq_len = 2 # TODO

def calculate_shift(
    image_seq_len,
    base_seq_len: int = 256,
    max_seq_len: int = 4096,
    base_shift: float = 0.5,
    max_shift: float = 1.15,
):
    m = (max_shift - base_shift) / (max_seq_len - base_seq_len)
    b = base_shift - m * base_seq_len
    mu = image_seq_len * m + b
    return mu

mu = calculate_shift(
    image_seq_len,
    pipe.scheduler.config.get("base_image_seq_len", 256),
    pipe.scheduler.config.get("max_image_seq_len", 4096),
    pipe.scheduler.config.get("base_shift", 0.5),
    pipe.scheduler.config.get("max_shift", 1.15),
)
timesteps, num_inference_steps = retrieve_timesteps(
    pipe.scheduler,
    num_inference_steps,
    device,
    sigmas=sigmas,
    mu=mu,
)

In [ ]:
########## FOR MULTIMODAL BLOCKS

# test = modules[0](X[0])
# print(test)

# from transformers import T5TokenizerT5TokenizerFast
# from transformers import T5TokenizerFast
block = pipe.transformer.transformer_blocks[0]
print(pipe)

timestep = timesteps[0].expand(X_img.shape[1]).to(X_text.dtype)

pooled_projections = pipe._get_clip_prompt_embeds(prompt)
encoder_hidden_states,pooled_projections,_ = pipe.encode_prompt(prompt)
guidance = None
encoder_hidden_states = pipe.transformer.context_embedder(encoder_hidden_states)

temb = (
            pipe.transformer.time_text_embed(timestep, pooled_projections)
            if guidance is None
            else pipe.transformer.time_text_embed(timestep, guidance, pooled_projections)
        )

# print("this shape")
# print(encoder_hidden_states.shape)
# print(pooled_projections.shape)

# enc_out, image_out = block(hidden_states=x, encoder_hidden_states=encoder_hidden_states, temb=temb)
enc_out, image_out = block(hidden_states=X_img[0], encoder_hidden_states=X_text[0], temb=temb)
print(f"output: {image_out}")

In [ ]:
def linearize(tfs, X_nom):
    """
    Linearize nonlinear dynamics f around nominal trajectory (X_nom, U_nom=0).

    Args:
        f: dynamics function f(x,u) -> x_next
        X_nom: nominal states (T+1, k, n)
        U_nom: nominal controls (T, m)

    Returns:
        A: linearized A matrices (T, n, n)
        B: linearized B matrices (T, n, m)
    """
    T = len(tfs)
    n = X_nom.shape[-1]
    # print(X_nom.shape)

    A = torch.zeros((T, n, n), dtype=X_nom.dtype, device=X_nom.device)
    # B = th.zeros((T, n, m), dtype=X_nom.dtype, device=X_nom.device)

    for t in range(T):
        x = X_nom[t].detach().requires_grad_(True)

        def f_last(x):
            xout = tfs[t](x)[1][..., -1, :]
            return xout.squeeze()
        # Compute Jacobians:
        Jx = torch.autograd.functional.jacobian(lambda x_: f_last(x_), x, create_graph=False, vectorize=True)   # shape: [n, *x.shape]
        # Ju = th.autograd.functional.jacobian(lambda u_: f_last(x, u_), u, create_graph=False, vectorize=True)   # shape: [n, *u.shape]
        print(Jx.shape)
        A[t] = Jx[...,-1,-1,:]
        # B[t] = Ju

    return A.detach().cpu()


def flux_block_wrapper(block, encoder_hidden_states, temb, x):
    return block(hidden_states=x, encoder_hidden_states=encoder_hidden_states, temb=temb)


In [ ]:
from functools import partial

wrapper_list = []
for i, tf in enumerate(pipe.transformer.transformer_blocks[1:]):
    wrapper_list.append(partial(flux_block_wrapper, tf, X_text[i], temb))

wrapper_list2 = []
for i, tf in enumerate(pipe.transformer.single_transformer_blocks[1:]):
    wrapper_list2.append(partial(flux_block_wrapper, tf, X_text_sing[i], temb))


# wrapper_list = [partial(flux_block_wrapper, tf, encoder_hidden_states, temb) for tf in pipe.transformer.transformer_blocks]

A = linearize(wrapper_list[:2], X_img[:2])
print(A.shape)


A_sing = linearize(wrapper_list2[:2], X_img_sing[:2])
print(A.shape)

